# Proyecto Final - Análisis de Datos 
# Formula 1
### 02 - Limpieza de Datos
**Integrantes:** 
- Esquivel, Kevin
- Simmons, Abigail
- Solis, Luis
- Villarreal, Sergio

**Fecha:** Julio 2026  

En este notebook cargamos los 12 archivos CSV de nuestra base de datos de Fórmula 1 
desde el Volume donde los subimos, y los registramos como tablas en Databricks 
para que los demás notebooks puedan usarlos.

In [0]:
# VERIFICACION PARA VALORES NULOS

# Reviso cuantos valores nulos tiene cada tabla en cada columna
# esto es importante para saber si hay datos faltantes que puedan afectar el analisis

from pyspark.sql.functions import col, sum as spark_sum, count, when, isnan

# las tablas que vamos a revisar
nombres_tablas = [
    "temporada", "circuito", "escuderia", "patrocinador",
    "piloto", "ingeniero", "carrera", "auto",
    "vuelta", "resultado", "tiempo_vuelta", "financiamiento"
]

print("REVISION DE VALORES NULOS POR TABLA\n")

for nombre in nombres_tablas:
    df = spark.table(nombre)
    
    # cuento los nulos en cada columna
    nulos = df.select([
        spark_sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]).collect()[0]
    
    # solo muestro las columnas que tienen al menos 1 nulo
    columnas_con_nulos = {c: nulos[c] for c in df.columns if nulos[c] > 0}
    
    if columnas_con_nulos:
        print(f"Tabla '{nombre}': tiene nulos en → {columnas_con_nulos}")
    else:
        print(f"Tabla '{nombre}': sin valores nulos ✓")

REVISION DE VALORES NULOS POR TABLA

Tabla 'temporada': sin valores nulos ✓
Tabla 'circuito': sin valores nulos ✓
Tabla 'escuderia': sin valores nulos ✓
Tabla 'patrocinador': sin valores nulos ✓
Tabla 'piloto': sin valores nulos ✓
Tabla 'ingeniero': tiene nulos en → {'id_supervisor': 21}
Tabla 'carrera': sin valores nulos ✓
Tabla 'auto': sin valores nulos ✓
Tabla 'vuelta': sin valores nulos ✓
Tabla 'resultado': sin valores nulos ✓
Tabla 'tiempo_vuelta': sin valores nulos ✓
Tabla 'financiamiento': sin valores nulos ✓


In [0]:
# VERIFICACION PARA VALORES DUPLICADOS

# verifico si hay filas completamente duplicadas en las tablas mas importantes
# las tablas con llave compuesta son las mas propensas a tener este problema

print("REVISION DE DUPLICADOS\n")

# para estas tablas reviso duplicados en la llave primaria completa
llaves = {
    "temporada":    ["id_temporada"],
    "circuito":     ["id_circuito"],
    "escuderia":    ["id_escuderia"],
    "piloto":       ["id_piloto"],
    "carrera":      ["id_carrera"],
    "resultado":    ["id_piloto", "id_carrera"],
    "tiempo_vuelta":["id_piloto", "id_carrera", "numero_vuelta"],
    "financiamiento":["id_patrocinador", "id_escuderia"],
    "vuelta":       ["id_carrera", "numero_vuelta"],
}

for nombre, columnas_llave in llaves.items():
    df = spark.table(nombre)
    total = df.count()
    distintos = df.dropDuplicates(columnas_llave).count()
    duplicados = total - distintos
    
    if duplicados > 0:
        print(f"Tabla '{nombre}': {duplicados} duplicados encontrados ⚠️")
    else:
        print(f"Tabla '{nombre}': sin duplicados ✓")

REVISION DE DUPLICADOS

Tabla 'temporada': sin duplicados ✓
Tabla 'circuito': sin duplicados ✓
Tabla 'escuderia': sin duplicados ✓
Tabla 'piloto': sin duplicados ✓
Tabla 'carrera': sin duplicados ✓
Tabla 'resultado': sin duplicados ✓
Tabla 'tiempo_vuelta': sin duplicados ✓
Tabla 'financiamiento': sin duplicados ✓
Tabla 'vuelta': sin duplicados ✓


In [0]:
# VERIFICACION DE TIPOS DE DATOS

# verifico que los tipos de datos de cada columna sean los correctos
# por ejemplo que los IDs sean enteros y no texto

print("TIPOS DE DATOS POR TABLA\n")

for nombre in nombres_tablas:
    df = spark.table(nombre)
    print(f"Tabla '{nombre}':")
    for campo, tipo in df.dtypes:
        print(f"   {campo}: {tipo}")
    print()

TIPOS DE DATOS POR TABLA

Tabla 'temporada':
   id_temporada: int
   anio: int
   carreras_programadas: int

Tabla 'circuito':
   id_circuito: int
   nombre: string
   ubicacion: string
   longitud_km: double
   numero_curvas: int

Tabla 'escuderia':
   id_escuderia: int
   nombre: string
   pais_origen: string
   director: string
   presupuesto: double

Tabla 'patrocinador':
   id_patrocinador: int
   nombre_empresa: string
   sector_industria: string

Tabla 'piloto':
   id_piloto: int
   nombre_piloto: string
   apellido_piloto: string
   nacionalidad: string
   fecha_nacimiento: date
   peso: double
   cantidad_campeonatos: int
   id_escuderia: int

Tabla 'ingeniero':
   id_ingeniero: int
   nombre_ingeniero: string
   apellido_ingeniero: string
   especialidad: string
   id_escuderia: int
   id_supervisor: int

Tabla 'carrera':
   id_carrera: int
   nombre: string
   fecha: date
   cantidad_vueltas: int
   temperatura: double
   probabilidad_lluvia: double
   id_circuito: int
   id

In [0]:
# VERIFICACION DE RANGOS NUMERICOS VALIDOS

# verifico que los valores numericos esten dentro de rangos que tengan sentido
# por ejemplo que las posiciones finales esten entre 1 y 20

from pyspark.sql.functions import min as spark_min, max as spark_max, avg as spark_avg

print("VERIFICACION DE RANGOS\n")

# Resultado: posiciones entre 1 y 20, puntos entre 0 y 25
resultado = spark.table("resultado")
print("Tabla 'resultado':")
resultado.select(
    spark_min("posicion_final").alias("posicion_minima"),
    spark_max("posicion_final").alias("posicion_maxima"),
    spark_min("puntos_obtenidos").alias("puntos_minimos"),
    spark_max("puntos_obtenidos").alias("puntos_maximos")
).show()

# Carrera: temperaturas entre 15 y 40, vueltas entre 50 y 78
carrera = spark.table("carrera")
print("Tabla 'carrera':")
carrera.select(
    spark_min("temperatura").alias("temp_minima"),
    spark_max("temperatura").alias("temp_maxima"),
    spark_min("cantidad_vueltas").alias("vueltas_minimas"),
    spark_max("cantidad_vueltas").alias("vueltas_maximas"),
    spark_min("probabilidad_lluvia").alias("lluvia_minima"),
    spark_max("probabilidad_lluvia").alias("lluvia_maxima")
).show()

# Piloto: peso entre 60 y 90, campeonatos entre 0 y 7
piloto = spark.table("piloto")
print("Tabla 'piloto':")
piloto.select(
    spark_min("peso").alias("peso_minimo"),
    spark_max("peso").alias("peso_maximo"),
    spark_min("cantidad_campeonatos").alias("campeonatos_minimos"),
    spark_max("cantidad_campeonatos").alias("campeonatos_maximos")
).show()

VERIFICACION DE RANGOS

Tabla 'resultado':
+---------------+---------------+--------------+--------------+
|posicion_minima|posicion_maxima|puntos_minimos|puntos_maximos|
+---------------+---------------+--------------+--------------+
|              1|             20|             0|            25|
+---------------+---------------+--------------+--------------+

Tabla 'carrera':
+-----------+-----------+---------------+---------------+-------------+-------------+
|temp_minima|temp_maxima|vueltas_minimas|vueltas_maximas|lluvia_minima|lluvia_maxima|
+-----------+-----------+---------------+---------------+-------------+-------------+
|       16.5|       33.5|             50|             76|         8.24|        72.56|
+-----------+-----------+---------------+---------------+-------------+-------------+

Tabla 'piloto':
+-----------+-----------+-------------------+-------------------+
|peso_minimo|peso_maximo|campeonatos_minimos|campeonatos_maximos|
+-----------+-----------+---------------

In [0]:
# RESUMEN FINAL DE LA CALIDAD DE LOS DATOS

# genero un resumen final que confirma que los datos estan listos para el analisis
# este resumen es lo que le pasamos al equipo de analisis exploratorio

print("RESUMEN FINAL DE CALIDAD DE DATOS\n")

for nombre in nombres_tablas:
    df = spark.table(nombre)
    filas = df.count()
    columnas = len(df.columns)
    print(f"✓ {nombre}: {filas} filas, {columnas} columnas — lista para analisis")

print("\nTodos los datos están verificados y listos para el notebook 03.")

RESUMEN FINAL DE CALIDAD DE DATOS

✓ temporada: 76 filas, 3 columnas — lista para analisis
✓ circuito: 70 filas, 5 columnas — lista para analisis
✓ escuderia: 21 filas, 5 columnas — lista para analisis
✓ patrocinador: 50 filas, 3 columnas — lista para analisis
✓ piloto: 50 filas, 8 columnas — lista para analisis
✓ ingeniero: 30 filas, 6 columnas — lista para analisis
✓ carrera: 120 filas, 8 columnas — lista para analisis
✓ auto: 50 filas, 8 columnas — lista para analisis
✓ vuelta: 7536 filas, 2 columnas — lista para analisis
✓ resultado: 2400 filas, 5 columnas — lista para analisis
✓ tiempo_vuelta: 75360 filas, 5 columnas — lista para analisis
✓ financiamiento: 135 filas, 3 columnas — lista para analisis

Todos los datos están verificados y listos para el notebook 03.
